In [1]:
from core.config import CURRENT
from core.clean import clean

import time
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support
from nltk.corpus import stopwords

# Loading
df = pd.read_csv(CURRENT["path"])
df["Text"] = df[CURRENT["text_col"]].apply(clean)
df = df.dropna(subset=[CURRENT["label_col"]])

# Binary label:
# 0 = rating 1,
# 1 = everything else
df["label_bin"] = (df[CURRENT["label_col"]] != 1).astype(int)

train, test = train_test_split(df, test_size=0.2, random_state=42)

vect = TfidfVectorizer(
    stop_words=stopwords.words('german'),
    max_features=CURRENT["max_features"],
)
X_train = vect.fit_transform(train["Text"])
X_test = vect.transform(test["Text"])

y_train = train["label_bin"]
y_test = test["label_bin"]

model = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)

start = time.time()
model.fit(X_train, y_train)
train_time = time.time() - start

y_pred = model.predict(X_test)

labels = sorted(set(y_test))
prec, rec, f1, sup = precision_recall_fscore_support(
    y_test, y_pred, labels=labels, zero_division=0
)

df_report = pd.DataFrame({
    "Precision": prec,
    "Recall": rec,
    "F1": f1,
    "Support": sup,
}, index=labels)

print(f"Train time: {train_time:.2f}s")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"F1 macro: {f1_score(y_test, y_pred, average='macro'):.4f}")
print()
print(df_report.round(4))

Train time: 1.56s
Accuracy: 0.8918
F1 macro: 0.8373

   Precision  Recall      F1  Support
0     0.9605  0.9042  0.9315    69908
1     0.6674  0.8380  0.7430    16040
